In this notebook other datasets are analysed and checked for their possible use in the project.

In [35]:
import pandas as pd

taxi_data = pd.read_parquet('../data/Taxi_Trips_cleaned.parquet')

In [37]:
display(taxi_data.head())

,trip_id,taxi_id,trip_start,trip_end,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare_usd,tips_usd,tolls_usd,extras_usd,trip_total_usd,payment_type,company,pickup_lat,pickup_lon,dropoff_lat,dropoff_lon
0,2e96dbb28bb463f8bc6151f00aeecbee8bcd4706,34766262f2e312774b1ad4651b99dc23b780dbd00b658f...,2026-05-01,2026-05-01 00:15:00,1269,13.04,56.0,39.0,33.00,7.7,0.0,5.0,46.20,credit card,taxicab insurance agency llc,41.792592,-87.769615,41.808916,-87.596183
1,2a8a4f2ebf353692695b453f97a76bd57aa6247a,0ee86e2a204cc2224e9ff2494686ee474cce6aba093a1b...,2026-05-01,2026-05-01 00:30:00,1923,21.48,76.0,NaN,52.00,15.0,0.0,36.5,104.00,credit card,sun taxi,41.979071,-87.903040,NaN,NaN
2,c53dc818bc94e87ad7dc88f3725edf99afdda4aa,d6e1a9e103336c396201abe9ceb00795fcd41e14ccbf54...,2026-05-01,2026-05-01 00:00:00,422,1.57,32.0,28.0,7.36,0.0,0.0,0.0,7.86,mobile,flash cab,41.878866,-87.625192,41.874005,-87.663518
3,83defb6f253341e24be3c2d010c6e253ba38ca47,03906d62f91d139ab93f74f50d1b208805b7fbb61e8d1e...,2026-05-01,2026-05-01 00:15:00,1083,9.93,76.0,NaN,26.25,0.0,0.0,5.0,31.25,cash,5 star taxi,41.980264,-87.913625,NaN,NaN
4,954c3b4006793e96c39ea56df7db1cf4283aede9,b5ad9d970e0745256fbb7517babae43abcb43115d076df...,2026-05-01,2026-05-01 00:30:00,1980,13.90,56.0,8.0,37.50,0.0,0.0,4.0,41.50,cash,transit administrative center inc,41.792592,-87.769615,41.899602,-87.633308


# Holidays
There are different type of holidays in the US, which affect different workgroups. 
- City Holidays: Local services of the city
- State Holidays: State government
- Bank Holidays: Money and Mail
- Corporate Holidays: Private sector

As different groups take different taxi rides, it is relevant to differentiate between the holiday type. Further it is necessary to divide between the work days. Through mobile working and new trends the trip behavior is most likely influenced, e.g. more people work from home in the beginning and ending of the week, also the behavior differs saturdays and sundays.

[Source: https://www.workingdays.us/mobile_home.php](https://www.workingdays.us/mobile_home.php)

## Preprocessing of holiday data

In [40]:
## Preprocessing of holiday data
holiday_data = pd.read_excel("../data/chicago_holidays.xlsx", sheet_name="Calendar")

# Cleanup
holiday_data.drop(columns=['HolidayName', 'HolidaySources', 'BankHolidayName', 
                           'CorporateHolidayName', 'CityHolidayName', 'StateHolidayName', 
                           'DayType'], inplace=True)
holiday_data.dropna(inplace=True)

# Casting
holiday_data['Date'] = pd.to_datetime(holiday_data['Date'], format='%d.%m.%Y')
holiday_data[['BankHoliday','CorporateHoliday','CityHoliday','StateHoliday']] = holiday_data[
    ['BankHoliday','CorporateHoliday','CityHoliday','StateHoliday']
].astype('int8')


# Dummy encoding
holiday_data = pd.get_dummies(holiday_data, columns=['DayOfWeek'], drop_first=True, dtype=int)

display(holiday_data.head())

,Date,BankHoliday,CorporateHoliday,CityHoliday,StateHoliday,DayOfWeek_Monday,DayOfWeek_Saturday,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday
0,2024-01-01,1,1,1,1,1,0,0,0,0,0
1,2024-01-02,0,0,0,0,0,0,0,0,1,0
2,2024-01-03,0,0,0,0,0,0,0,0,0,1
3,2024-01-04,0,0,0,0,0,0,0,1,0,0
4,2024-01-05,0,0,0,0,0,0,0,0,0,0


In [34]:
holiday_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 852 entries, 0 to 851
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 852 non-null    datetime64[us]
 1   BankHoliday          852 non-null    int8          
 2   CorporateHoliday     852 non-null    int8          
 3   CityHoliday          852 non-null    int8          
 4   StateHoliday         852 non-null    int8          
 5   DayOfWeek_Monday     852 non-null    int64         
 6   DayOfWeek_Saturday   852 non-null    int64         
 7   DayOfWeek_Sunday     852 non-null    int64         
 8   DayOfWeek_Thursday   852 non-null    int64         
 9   DayOfWeek_Tuesday    852 non-null    int64         
 10  DayOfWeek_Wednesday  852 non-null    int64         
dtypes: datetime64[us](1), int64(6), int8(4)
memory usage: 50.1 KB


## Merging holiday data into taxi data

In [45]:
taxi_data["date_day"] = taxi_data["trip_start"].dt.floor("D")

holiday_data = holiday_data.rename(columns={"Date": "date_day"})

taxi_data_copy = taxi_data.merge(
    holiday_data,
    on="date_day",
    how="left"
)#.drop(columns=["date_day"])

display(taxi_data_copy.head())

,trip_id,taxi_id,trip_start,trip_end,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare_usd,tips_usd,...,BankHoliday,CorporateHoliday,CityHoliday,StateHoliday,DayOfWeek_Monday,DayOfWeek_Saturday,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday
0,2e96dbb28bb463f8bc6151f00aeecbee8bcd4706,34766262f2e312774b1ad4651b99dc23b780dbd00b658f...,2026-05-01,2026-05-01 00:15:00,1269,13.04,56.0,39.0,33.00,7.7,...,0,0,0,0,0,0,0,0,0,0
1,2a8a4f2ebf353692695b453f97a76bd57aa6247a,0ee86e2a204cc2224e9ff2494686ee474cce6aba093a1b...,2026-05-01,2026-05-01 00:30:00,1923,21.48,76.0,NaN,52.00,15.0,...,0,0,0,0,0,0,0,0,0,0
2,c53dc818bc94e87ad7dc88f3725edf99afdda4aa,d6e1a9e103336c396201abe9ceb00795fcd41e14ccbf54...,2026-05-01,2026-05-01 00:00:00,422,1.57,32.0,28.0,7.36,0.0,...,0,0,0,0,0,0,0,0,0,0
3,83defb6f253341e24be3c2d010c6e253ba38ca47,03906d62f91d139ab93f74f50d1b208805b7fbb61e8d1e...,2026-05-01,2026-05-01 00:15:00,1083,9.93,76.0,NaN,26.25,0.0,...,0,0,0,0,0,0,0,0,0,0
4,954c3b4006793e96c39ea56df7db1cf4283aede9,b5ad9d970e0745256fbb7517babae43abcb43115d076df...,2026-05-01,2026-05-01 00:30:00,1980,13.90,56.0,8.0,37.50,0.0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
taxi_data_copy.columns

Index(['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds',
       'trip_miles', 'pickup_community_area', 'dropoff_community_area',
       'fare_usd', 'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd',
       'payment_type', 'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat',
       'dropoff_lon', 'BankHoliday', 'CorporateHoliday', 'CityHoliday',
       'StateHoliday', 'DayOfWeek_Monday', 'DayOfWeek_Saturday',
       'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday',
       'DayOfWeek_Wednesday'],
      dtype='str')

# Events
Events are typical destination for and also from Taxi trips. It is expected that not all people travel by public transport or their own cars to concerts, sports matches and other events. The city of Chicago offers a list of such events including their coordinates: https://data.cityofchicago.org/Events/Special-Events/xgse-8eg7/about_data

The data offers the ward in which the event is taking place as well as the exact location. Therefore the data can be mapped to the taxi trips and locations accordingly.

In [56]:
event_data = pd.read_csv("../data/chicago_events_20260515.csv")
event_data.drop(columns=['Venue', ], inplace=True)

display(event_data)

,Date,Venue Address,Event Type,Event Details,Start Time,Ward,Location
0,11/21/2025,50 W. Washington Street,Special Event,Christkindlmarket - Chicago,11:00AM,42,POINT (-87.6302 41.88325)
1,11/22/2025,50 W. Washington Street,Special Event,Christkindlmarket - Chicago,11:00AM,42,POINT (-87.6302 41.88325)
2,11/23/2025,50 W. Washington Street,Special Event,Christkindlmarket - Chicago,11:00AM,42,POINT (-87.6302 41.88325)
3,11/24/2025,50 W. Washington Street,Special Event,Christkindlmarket - Chicago,11:00AM,42,POINT (-87.6302 41.88325)
4,11/25/2025,50 W. Washington Street,Special Event,Christkindlmarket - Chicago,11:00AM,42,POINT (-87.6302 41.88325)
...,...,...,...,...,...,...,...
590,05/22/2026,3300-3359 W CARMEN AVE,Block Party,Von Steuben Senior Celebration,NaN,39,POINT (-87.71198 41.97394)
591,06/06/2026,3600-3800 W MADISON ST\n3500-3600 W MADISON ST...,Athletic,Bank of America Chicago 13.1,NaN,28,NaN
592,06/07/2026,3600-3800 W MADISON ST\n3500-3600 W MADISON ST...,Athletic,Bank of America Chicago 13.1,NaN,28,NaN
593,06/20/2026,1305-1325 N ASTOR ST,Block Party,1300 Astor BP,NaN,43,POINT (-87.62744 41.90648)


## Cleanup
Below the event data is cleaned up. 

The column "Start Time" contains times when the event will start, especially three values are special and are removed or changed:
- "ALL DAY": Will be set to 9 AM as it is expected that these events begin early in the morning
- "nan"/"N/A": ?
- "TBD": ?

In [ ]:
print('Unique values of start time:')
print(event_data['Start Time'].unique())
print('\nEvents with TBA or NaN start time:')
print(len(event_data[event_data['Start Time'].isin(['TBA', pd.NA, 'N/A'])].index))

Unique values of start time:
<ArrowStringArray>
['11:00AM',     'TBA',  '4:30PM',  '6:00AM', 'ALL DAY',       nan,  '7:30PM',
 '12:00PM',  '3:00PM',  '7:00PM',  '1:30PM',  '8:00PM',  '7:45PM',  '8:30PM',
  '6:00PM',  '2:30PM',  '6:30PM',  '6:45PM', '10:30AM',  '5:30PM',  '2:00PM',
  '5:00PM',  '3:30PM',  '1:00PM',  '4:00PM', '11:00am']
Length: 26, dtype: str

Events with TBA or NaN start time:
309


In [ ]:
event_data['Start Time'] = event_data['Start Time'].replace('ALL DAY', '9:00 AM')